<a href="https://colab.research.google.com/github/quarcs-lab/geo4cambodia/blob/main/notebooks/expdpy_analyze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Panel econometrics for Cambodia with `geo4cambodia` + `expdpy`

Part of the [GeoDevelopment Observatory of Cambodia](https://quarcs-lab.github.io/geo4cambodia/).

**[geo4cambodia](https://github.com/quarcs-lab/geo4cambodia)** prepares the panels; **[expdpy](https://cmg777.github.io/expdpy/)** estimates panel models: pooled, between, fixed and random effects, Hausman tests, two-way fixed effects with clustered standard errors, and convergence. Every result reads itself out with `.interpret()`.

Four questions, four sections:

| | Question | Panel |
| --- | --- | --- |
| 1 | Do night-time lights track economic activity? | 25 provinces, 2000–2019; 17 GDL regions, 2013–2019 |
| 2 | What goes with human development? | 17 GDL regions, 2013–2019 |
| 3 | Are districts converging, and does space matter? | 202 districts, 1990–2022 (with geometrics) |
| 4 | Is pollution higher where development is higher? | 25 provinces, 2017–2022 |

The results are **associations**, not causal effects. The exploratory companion is [`expdpy_explore.ipynb`](https://colab.research.google.com/github/quarcs-lab/geo4cambodia/blob/main/notebooks/expdpy_explore.ipynb).

## 0. Setup

In [1]:
import importlib.util

missing = [p for p in ("geo4cambodia", "expdpy", "geometrics") if importlib.util.find_spec(p) is None]
if missing:
    !pip install -q {" ".join(missing)}

In [2]:
import warnings

import numpy as np
import pandas as pd
import geo4cambodia as geo
import expdpy as ex

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Numba not installed")   # harmless: mapclassify falls back to pure Python

## 1. Do night-time lights track economic activity?

Night-time lights are widely used as a proxy for economic activity where GDP statistics are scarce. We relate them to income twice:

- **Provinces, 2000–2019:** electricity-based GDP and VIIRS-like lights. Caveat: the electricity-based GDP series is itself *built* from calibrated night-time lights (Chen et al., 2022), so this is partly a consistency check between two satellite products.
- **GDL regions, 2013–2019:** gross national income from Global Data Lab, which comes from surveys and national accounts, and DMSP-like lights. This is the more independent test.

Some rural provinces show **zero lights** in the 2000s, so we use the inverse hyperbolic sine `asinh(lights)`, which behaves like a log for large values and is defined at zero.

In [3]:
env, env_def = geo.for_expdpy("env", years=(2000, 2019))
p = env.dropna(subset=["total_EGDP", "total_VNL"]).copy()
p["ln_egdp"] = np.log(p["total_EGDP"])
p["asinh_lights"] = np.arcsinh(p["total_VNL"])
p["ln_pop"] = np.log(p["total_GlobPOP_Count"])
p, p_def = geo.for_expdpy(p[["geo_id", "year", "ln_egdp", "asinh_lights", "ln_pop"]],
                          labels={"ln_egdp": "Log electricity-based GDP", "asinh_lights": "asinh(night-time lights)",
                                  "ln_pop": "Log population"})
p = ex.set_labels(p, p_def, set_panel=True)

`analyze_panel_table` fits four estimators side by side. **Pooled** OLS uses all variation; **between** uses only differences in province averages; **fixed effects** (FE) only changes within each province over time; **random effects** (RE) a weighted mix.

In [4]:
models = ex.analyze_panel_table(p, "ln_egdp", ["asinh_lights", "ln_pop"])
print(models.interpret())
models.etable

This OLS regression relates **ln_egdp** to its regressors. Standard errors are clustered by *geo_id*.

- **asinh_lights**: each one-unit increase is associated with ln_egdp that is 0.137 higher (statistically significant at the 1% level).
- **ln_pop**: each one-unit increase is associated with ln_egdp that is 0.283 higher (not statistically significant at conventional levels).

Model fit: N = 500, R² = 0.269.
(This reads the first of 4 models in the table.)

_These are associations, not causal effects. A causal reading needs a research design — see `explain('correlation_vs_causation')`._


,Pooled OLS,Between,Fixed effects,Random effects
Dep. Variable,ln_egdp,ln_egdp,ln_egdp,ln_egdp
Estimator,PooledOLS,BetweenOLS,PanelOLS,RandomEffects
No. Observations,500,25,500,500
Cov. Est.,Clustered,Unadjusted,Clustered,Clustered
R-squared,0.2694,0.2269,0.7389,0.7305
R-Squared (Within),0.7264,0.7116,0.7389,0.7389
R-Squared (Between),0.2264,0.2269,0.9793,0.1964
R-Squared (Overall),0.2694,0.2686,0.9787,0.2431
F-statistic,91.637,3.2277,669.28,673.52
P-value (F-stat),0.0000,0.0590,0.0000,0.0000


The **Hausman test** asks whether FE and RE estimates differ systematically. If they do, RE's assumption (province effects uncorrelated with the regressors) fails and FE is preferred.

In [5]:
print(ex.analyze_hausman_test(p, "ln_egdp", ["asinh_lights", "ln_pop"]).interpret())

Hausman test (χ²(2) = 0.855, p = 0.6522): **fail to reject** the null — **random effects** is admissible (and more efficient than fixed effects). Note that failing to reject reflects a lack of evidence against random effects, not proof that it is correct.


**Two-way fixed effects** add year effects, which absorb shocks common to all provinces (national growth, sensor changes), with standard errors clustered by province:

In [6]:
twfe = ex.analyze_regression_table(p, dvs="ln_egdp", idvs=["asinh_lights", "ln_pop"],
                                   feffects=["geo_id", "year"], clusters=["geo_id"])
print(twfe.interpret())
twfe.etable

This OLS regression relates **ln_egdp** to its regressors. Fixed effects for *geo_id + year* absorb time-invariant differences, so coefficients reflect variation **within** each group. Standard errors are clustered by *geo_id*.

- **asinh_lights**: each one-unit increase is associated with ln_egdp that is 0.0035 higher (not statistically significant at conventional levels).
- **ln_pop**: each one-unit increase is associated with ln_egdp that is 0.0315 lower (not statistically significant at conventional levels).

Model fit: N = 500, R² = 0.999, within-R² = 0.0178.

_These are associations, not causal effects. A causal reading needs a research design — see `explain('correlation_vs_causation')`._


GT(_tbl_data=  __index_level_0__         __index_level_1__                    0
0              coef  asinh(night-time lights)   0.004 <br> (0.005)
1              coef            Log population  -0.031 <br> (0.030)
2                fe                    geo ID                    -
3                fe                      Year                    -
4             stats              Observations                  500
5             stats                        R²                0.999, _body=<great_tables._gt_data.Body object at 0x1260658d0>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align=None, column_width=None, stub_level=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None, stub_level=0), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x12604fa90>, _spanners=Spanners([SpannerInfo(spanner_id='Log electricity-based GDP', spanner_level=1, spanner_label=Html(text='Log electricity-based GDP'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12604f910>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12604f250>, _source_notes=[Html(text='Significance levels: * p &lt; 0.05, ** p &lt; 0.01, *** p &lt; 0.001. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x12604e890>, _formats=[<great_tables._gt_data.FormatInfo object at 0x126069b90>, <great_tables._gt_data.FormatInfo object at 0x1265df1d0>, <great_tables._gt_data.FormatInfo object at 0x1265dfc90>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, cat

In [7]:
ex.analyze_coefficient_plot(twfe).fig

**Reading the results.** Within a province, brighter years go with higher electricity-based GDP (FE coefficient ≈ 0.12, highly significant), and the Hausman test does not reject random effects. Once **year effects** absorb what all provinces share, the lights coefficient falls to almost zero. The within-province association is mostly a common national trend: lights and GDP both grew everywhere. Province-specific deviations from that trend are not matched. This is a typical lesson of two-way fixed effects.

**The independent test: income and lights across GDL regions.** Global Data Lab's GNI does not come from satellites:

In [8]:
gdl, gdl_def = geo.for_expdpy("hdi")
gdl = ex.set_labels(gdl, gdl_def, set_panel=True)
income = ex.analyze_panel_table(gdl, "lnGNI", ["lnNTL"])
print(income.interpret())
income.etable

This OLS regression relates **lnGNI** to its regressors. Standard errors are clustered by *geo_id*.

- **lnNTL**: each one-unit increase is associated with lnGNI that is 0.0572 higher (not statistically significant at conventional levels).

Model fit: N = 119, R² = 0.00686.
(This reads the first of 4 models in the table.)

_These are associations, not causal effects. A causal reading needs a research design — see `explain('correlation_vs_causation')`._


,Pooled OLS,Between,Fixed effects,Random effects
Dep. Variable,lnGNI,lnGNI,lnGNI,lnGNI
Estimator,PooledOLS,BetweenOLS,PanelOLS,RandomEffects
No. Observations,119,17,119,119
Cov. Est.,Clustered,Unadjusted,Clustered,Clustered
R-squared,0.0069,0.0010,0.6194,0.5850
R-Squared (Within),0.4274,-0.3484,0.6194,0.6194
R-Squared (Between),-0.0067,0.0010,0.1653,-0.0240
R-Squared (Overall),0.0069,-0.0099,0.1653,-0.0039
F-statistic,0.8081,0.0151,164.40,164.92
P-value (F-stat),0.3705,0.9040,0.0000,0.0000


**Reading the results.** Across regions (pooled and between), lights say almost nothing about income levels (R² ≈ 0). Within a region over time, income and lights rise together (FE ≈ 0.13). With seven years and no year effects, this again mixes regional dynamics with national growth. For ranking Cambodian regions by income, DMSP-like lights are a weak proxy.

## 2. What goes with human development?

Across Cambodia's 17 GDL regions, is the Human Development Index higher where lights per person are brighter? Here we also bring in province data: `geo.combine()` sums the provinces' population to GDL regions, using the aggregation rules in the metadata.

In [9]:
hd = geo.combine({"hdi": None, "env": ["total_GlobPOP_Count"]}, scale="gdl", years=(2013, 2019))
hd["ln_pop"] = np.log(hd["total_GlobPOP_Count"])
hd, hd_def = geo.for_expdpy(hd[["geo_id", "year", "HDI", "lnNTLpc", "ln_pop"]],
                            labels={"ln_pop": "Log population (gridded, summed from provinces)"})
hd = ex.set_labels(hd, hd_def, set_panel=True)

hdi_models = ex.analyze_panel_table(hd, "HDI", ["lnNTLpc", "ln_pop"])
print(hdi_models.interpret())
hdi_models.etable

This OLS regression relates **HDI** to its regressors. Standard errors are clustered by *geo_id*.

- **lnNTLpc**: each one-unit increase is associated with HDI that is 0.00205 lower (not statistically significant at conventional levels).
- **ln_pop**: each one-unit increase is associated with HDI that is 0.0159 higher (not statistically significant at conventional levels).

Model fit: N = 119, R² = 0.0715.
(This reads the first of 4 models in the table.)

_These are associations, not causal effects. A causal reading needs a research design — see `explain('correlation_vs_causation')`._


,Pooled OLS,Between,Fixed effects,Random effects
Dep. Variable,HDI,HDI,HDI,HDI
Estimator,PooledOLS,BetweenOLS,PanelOLS,RandomEffects
No. Observations,119,17,119,119
Cov. Est.,Clustered,Unadjusted,Clustered,Clustered
R-squared,0.0715,0.1595,0.7246,0.6662
R-Squared (Within),-0.1575,-3.7282,0.7246,0.7104
R-Squared (Between),0.0869,0.1595,-1.8792,-0.2965
R-Squared (Overall),0.0715,-0.0845,-1.8781,-0.2333
F-statistic,4.4679,1.3284,131.58,115.77
P-value (F-stat),0.0135,0.2963,0.0000,0.0000


In [10]:
print(ex.analyze_hausman_test(hd, "HDI", ["lnNTLpc", "ln_pop"]).interpret())

Hausman test (χ²(2) = 9.534, p = 0.008506): **reject** the null — the random-effects assumption is violated, so prefer **fixed effects**. Note that failing to reject reflects a lack of evidence against random effects, not proof that it is correct.


**Reading the results.** Across regions (pooled, between), neither lights per person nor population is associated with HDI. Within regions over time, HDI rises with both (FE: lights per person ≈ 0.008, population ≈ 0.10). The Hausman test **rejects** random effects, because the region effects are correlated with the regressors, so fixed effects are the appropriate estimator. With seven years, these within-region associations largely capture the joint upward trends of 2013–2019.

## 3. Are districts converging, and does space matter?

**β-convergence** asks whether districts that were poorer in 1990 grew faster; **σ-convergence** whether the spread of log incomes narrowed. expdpy estimates both without space. We then compare with [geometrics](https://quarcs-lab.github.io/geometrics/), which models spatial spillovers between neighbouring districts.

In [11]:
dist, dist_def = geo.for_expdpy("gdppc", groups=["province"])
dist = ex.set_labels(dist, dist_def, set_panel=True)

beta = ex.analyze_beta_convergence(dist, "ln_y", rolling=False)
print(beta.interpret())
beta.fig

Across 202 units over a 32-period horizon, the average growth rate of **ln_y** is negatively associated with its initial level (β = -0.0097). Units that start lower tend to grow faster — the pattern of **unconditional β-convergence**.
The implied speed of convergence is λ = 0.0116 per period, so about half of an initial gap closes every 59.7 periods (the half-life).

_These are associations, not causal effects. A causal reading needs a research design — see `explain('correlation_vs_causation')`._

In [12]:
sigma = ex.analyze_sigma_convergence(dist, "ln_y")
print(sigma.interpret())
sigma.fig

Across 202 units over 33 periods, the cross-sectional standard deviation of **ln_y** narrowed (from 0.439 to 0.394). The log-dispersion trend is -0.00128 per period — about 0.128% less dispersion each period (not statistically significant at conventional levels) — the pattern of **σ-convergence**.
The Gini index also narrowed over the same span (trend -0.00695 per period).

_These are associations, not causal effects. A causal reading needs a research design — see `explain('correlation_vs_causation')`._


In [13]:
clubs = ex.analyze_convergence_clubs(dist, "ln_y")
print(clubs.interpret())
clubs.fig

Across 202 units over 33 periods, the Phillips-Sul log(t) test for **ln_y** rejects global convergence (t = -9.8 <= -1.65). The clustering algorithm splits the panel into **2 convergence clubs** — groups that each converge internally but not with one another: Club 1 (63), Club 2 (139).
Club 1 collects the highest-ranked units; within each club the log(t) slope b = 2*alpha is positive enough that its t-statistic clears -1.65.

_These are associations, not causal effects. A causal reading needs a research design — see `explain('correlation_vs_causation')`._


**Non-spatial vs spatial.** geometrics needs boundaries and spatial weights, which `geo.for_geometrics()` provides. Note the convention: geometrics takes **levels** (`"y"`) and logs them itself, while expdpy takes the variable you want analysed, here already in logs (`"ln_y"`). Done this way, both estimate the same OLS β.

In [14]:
import geometrics as gm

gdf, gdist, gdist_dict = geo.for_geometrics("gdppc")
gdist = gm.set_labels(gdist, gdist_dict, set_panel=True)
w = gm.make_weights(gdf, method="queen")

ols = gm.analyze_beta_convergence(gdist, "y")
sdm = gm.analyze_beta_convergence(gdist, "y", gdf=gdf, w=w, model="sdm")

comparison = pd.DataFrame({
    "model": ["expdpy: OLS (non-spatial)", "geometrics: OLS (non-spatial)", "geometrics: spatial Durbin, total effect"],
    "beta": [beta.beta, ols.beta_total, sdm.beta_total],
    "half-life (years)": [beta.half_life, ols.half_life, sdm.half_life],
}).round(4)
comparison

,model,beta,half-life (years)
0,expdpy: OLS (non-spatial),-0.0097,59.7158
1,geometrics: OLS (non-spatial),-0.0097,59.7158
2,"geometrics: spatial Durbin, total effect",-0.0023,296.1693


In [15]:
print(sdm.interpret())

Across 202 units, the growth of **y** over a 32-period window is associated with its initial log level with a total slope of **-0.00225** (SE 0.00325), not statistically significant at conventional levels.
The slope is negative — the β-convergence pattern: units that started lower tended to grow faster, narrowing initial gaps.
That slope implies a convergence speed of λ = 0.00234 per period and a half-life of about 296 periods — the time for half of an initial gap to close at this pace.
The SDM decomposition splits the total into a direct component of -0.0169 (own initial level) and a spillover (indirect) component of 0.0146 operating through neighboring units, under the weights: queen contiguity, row-standardized, n=202.
The spatial-lag parameter ρ = 0.513 says each unit's growth moves together with its neighbors' growth, so part of the convergence pattern is shared across space rather than purely unit-by-unit.

_These are associations, not causal effects. A causal reading needs a res

**When does space matter? Here, a lot.** Both packages agree on the non-spatial β (−0.0097, a half-life of about 60 years). Once neighbours enter the model, the total effect in the spatial Durbin model shrinks to about −0.002 and is no longer statistically significant. Income in Cambodia is strongly clustered: Moran's I of log GDP per capita is about 0.6. Much of the apparent catch-up therefore reflects **regional** dynamics, whole neighbourhoods of districts growing together, rather than each poor district closing its own gap. A convergence speed estimated without space would overstate how fast individual districts catch up.

## 4. Is pollution higher where development is higher?

Fine particulate matter (PM2.5) data cover 2017–2022. As in the explore notebook, the mean concentration is approximated as the zonal sum over the number of 0.01° grid cells in each province. Development is measured by lights per person. We fit a quadratic, the shape of an *environmental Kuznets curve*: pollution rising with development and then falling. The first model uses year effects only, so it compares provinces. The second adds province fixed effects, so it uses only changes within provinces.

In [16]:
env_all, _ = geo.for_expdpy("env", years=(2017, 2022))
provinces = geo.boundaries("province").merge(geo.units("province")[["geo_id", "area_km2"]], on="geo_id")
lat = provinces.to_crs("EPSG:32648").centroid.to_crs("EPSG:4326").y
cells = dict(zip(provinces["geo_id"], provinces["area_km2"].astype(float) / ((111.32 * 0.01) ** 2 * np.cos(np.radians(lat)))))

q = env_all.dropna(subset=["total_GHAP", "total_VNL"]).copy()
q["pm25"] = q["total_GHAP"] / q["geo_id"].map(cells)
q["ln_lights_pc"] = np.log(q["total_VNL"] / q["total_GlobPOP_Count"] * 1000)
q["ln_lights_pc_sq"] = q["ln_lights_pc"] ** 2
q, q_def = geo.for_expdpy(q[["geo_id", "year", "pm25", "ln_lights_pc", "ln_lights_pc_sq"]],
                          labels={"pm25": "Mean PM2.5 (µg/m³, approx.)", "ln_lights_pc": "Log lights per 1,000 people",
                                  "ln_lights_pc_sq": "Log lights per 1,000 people, squared"})
q = ex.set_labels(q, q_def, set_panel=True)

across = ex.analyze_regression_table(q, dvs="pm25", idvs=["ln_lights_pc", "ln_lights_pc_sq"],
                                     feffects=["year"], clusters=["geo_id"])
print(across.interpret())
across.etable

This OLS regression relates **pm25** to its regressors. Fixed effects for *year* absorb time-invariant differences, so coefficients reflect variation **within** each group. Standard errors are clustered by *geo_id*.

- **ln_lights_pc**: each one-unit increase is associated with pm25 that is 0.251 lower (not statistically significant at conventional levels).
- **ln_lights_pc_sq**: each one-unit increase is associated with pm25 that is 0.137 higher (not statistically significant at conventional levels).

Model fit: N = 150, R² = 0.171, within-R² = 0.0276.

_These are associations, not causal effects. A causal reading needs a research design — see `explain('correlation_vs_causation')`._


GT(_tbl_data=  __index_level_0__                     __index_level_1__                    0
0              coef           Log lights per 1,000 people  -0.251 <br> (0.652)
1              coef  Log lights per 1,000 people, squared   0.137 <br> (0.323)
2                fe                                  Year                    x
3             stats                          Observations                  150
4             stats                                    R²                0.171, _body=<great_tables._gt_data.Body object at 0x12a3ea5d0>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align=None, column_width=None, stub_level=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None, stub_level=0), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x12a3ed210>, _spanners=Spanners([SpannerInfo(spanner_id='Mean PM2.5 (µg/m³, approx.)', spanner_level=1, spanner_label=Html(text='Mean PM2.5 (µg/m³, approx.)'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12a3eaf10>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12a3ea950>, _source_notes=[Html(text='Significance levels: * p &lt; 0.05, ** p &lt; 0.01, *** p &lt; 0.001. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x127ecbcd0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x12a3bc410>, <great_tables._gt_data.FormatInfo object at 0x12a3e68d0>, <great_tables._gt_data.FormatInfo object at 0x12a3e5010>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=

In [17]:
within = ex.analyze_regression_table(q, dvs="pm25", idvs=["ln_lights_pc", "ln_lights_pc_sq"],
                                     feffects=["geo_id", "year"], clusters=["geo_id"])
print(within.interpret())
within.etable

This OLS regression relates **pm25** to its regressors. Fixed effects for *geo_id + year* absorb time-invariant differences, so coefficients reflect variation **within** each group. Standard errors are clustered by *geo_id*.

- **ln_lights_pc**: each one-unit increase is associated with pm25 that is 0.244 lower (not statistically significant at conventional levels).
- **ln_lights_pc_sq**: each one-unit increase is associated with pm25 that is 0.0235 lower (not statistically significant at conventional levels).

Model fit: N = 150, R² = 0.95, within-R² = 0.0148.

_These are associations, not causal effects. A causal reading needs a research design — see `explain('correlation_vs_causation')`._


GT(_tbl_data=  __index_level_0__                     __index_level_1__                    0
0              coef           Log lights per 1,000 people  -0.244 <br> (0.150)
1              coef  Log lights per 1,000 people, squared  -0.023 <br> (0.046)
2                fe                                geo ID                    -
3                fe                                  Year                    -
4             stats                          Observations                  150
5             stats                                    R²                 0.95, _body=<great_tables._gt_data.Body object at 0x12a3f25d0>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align=None, column_width=None, stub_level=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None, stub_level=0), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x12a41d750>, _spanners=Spanners([SpannerInfo(spanner_id='Mean PM2.5 (µg/m³, approx.)', spanner_level=1, spanner_label=Html(text='Mean PM2.5 (µg/m³, approx.)'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12a3f2950>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12a3f3790>, _source_notes=[Html(text='Significance levels: * p &lt; 0.05, ** p &lt; 0.01, *** p &lt; 0.001. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x12a3f3350>, _formats=[<great_tables._gt_data.FormatInfo object at 0x12a41ce50>, <great_tables._gt_data.FormatInfo object at 0x12a41d110>, <great_tables._gt_data.FormatInfo object at 0x12a41d390>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, catego

In [18]:
# Where the fitted curve peaks (or bottoms out): -b1 / (2 * b2), in log lights per 1,000 people
coef = across.tidy().set_index("term")["Estimate"]
turning = -coef["ln_lights_pc"] / (2 * coef["ln_lights_pc_sq"])
shape = "peaks" if coef["ln_lights_pc_sq"] < 0 else "bottoms out"
print(f"Across provinces the fitted curve {shape} at {np.exp(turning):.1f} lights per 1,000 people "
      f"(observed range {np.exp(q['ln_lights_pc'].min()):.1f}–{np.exp(q['ln_lights_pc'].max()):.1f}).")

Across provinces the fitted curve bottoms out at 2.5 lights per 1,000 people (observed range 0.1–19.3).


**Reading the results.** Neither specification finds a statistically significant association between PM2.5 and lights per person. Across provinces the point estimates even trace a U-shape, the opposite of an environmental Kuznets curve. In these data PM2.5 rather follows population (r ≈ 0.64 in the explore notebook) and regional haze. With 25 provinces and six years, the estimates are fragile. Treat them as a starting point, for example for district-level PM2.5 built with `geo.zonal()`.

## 5. Learn the concepts

expdpy explains every method in plain language:

In [19]:
ex.explain("hausman")

### Hausman test (fixed vs random effects)

**What it is.** The Hausman test compares the fixed-effects and random-effects coefficient estimates. Under the random-effects assumption both are consistent and should agree; a large difference signals that the unit effects are correlated with the regressors, so random effects is biased.

**When to use it.** To choose between fixed and random effects: reject the null (large statistic, small p-value) and prefer fixed effects; fail to reject and random effects is admissible (and more efficient).

**Watch out for.**
- Failing to reject is not proof that random effects is correct — it may just be low power.
- The covariance difference can be non-positive-definite in finite samples; a generalized inverse is used, which makes the statistic approximate.

*See also:* fixed_effects, random_effects

*References:* Hausman (1978); Wooldridge, Introductory Econometrics, ch. 14

Other topics: `ex.list_topics()`. Interactive sandboxes such as `ex.learn_pooled_vs_fixed_effects()` simulate data from a known truth.

## Next steps

- Explore the panels first: [`expdpy_explore.ipynb`](https://colab.research.google.com/github/quarcs-lab/geo4cambodia/blob/main/notebooks/expdpy_explore.ipynb)
- Spatial analysis with geometrics: [`geometrics_convergence.ipynb`](https://colab.research.google.com/github/quarcs-lab/geo4cambodia/blob/main/notebooks/geometrics_convergence.ipynb)
- Bring your own indicators (`geo.link`, `geo.zonal`, `geo.combine`): [quickstart](https://colab.research.google.com/github/quarcs-lab/geo4cambodia/blob/main/notebooks/geo4cambodia_quickstart.ipynb)
- No code: expdpy's [Analyze app](https://expdpy-analyze.streamlit.app)